In [30]:
import pandas as pd
import numpy as np
import torch

from pytorch_forecasting import TimeSeriesDataSet, TemporalFusionTransformer
from pytorch_forecasting.metrics import CrossEntropy

from lightning.pytorch import Trainer
from lightning.pytorch.callbacks import EarlyStopping

In [31]:
import torch
torch.set_float32_matmul_precision("high")

print("GPU Available:", torch.cuda.is_available())

GPU Available: True


In [32]:
df = pd.read_csv("../data/features/features.csv")

df = df.sort_values(["patient_id", "hour_from_admission"]).reset_index(drop=True)

TARGET = "deterioration_next_12h"

In [33]:
fold = 0

train_df = pd.read_csv(f"../data/splits/train_fold_{fold}.csv")
val_df = pd.read_csv(f"../data/splits/val_fold_{fold}.csv")

In [ ]:
# 🔥 SAMPLE ONLY 40% DATA (SAFE)
train_df = train_df.sample(frac=0.4, random_state=42)
val_df = val_df.sample(frac=0.4, random_state=42)

print(len(train_df), len(val_df))

In [34]:
def clean_dataframe(df):
    df = df.copy()
    
    for col in df.columns:
        if df[col].dtype == "object":
            df[col] = df[col].replace(["none", "None", "nan"], np.nan)
            df[col] = df[col].astype("category").cat.codes
    
    df = df.apply(pd.to_numeric, errors='coerce')
    df = df.fillna(0)
    
    return df

train_df = clean_dataframe(train_df)
val_df = clean_dataframe(val_df)

In [35]:
def add_time_idx(df):
    df = df.sort_values(["patient_id", "hour_from_admission"]).copy()
    df["time_idx"] = df.groupby("patient_id").cumcount()
    return df

train_df = add_time_idx(train_df)
val_df = add_time_idx(val_df)

In [37]:
def fix_tft_categoricals(df):
    df = df.copy()
    
    categorical_cols = ["gender", "admission_type"]
    
    for col in categorical_cols:
        df[col] = df[col].astype(str)
        df[col] = df[col].astype("category")
    
    return df

train_df = fix_tft_categoricals(train_df)
val_df = fix_tft_categoricals(val_df)

In [38]:
train_df[TARGET] = train_df[TARGET].astype(int)
val_df[TARGET] = val_df[TARGET].astype(int)

In [ ]:
static_categoricals = ["gender", "admission_type"]
static_reals = ["age", "comorbidity_index"]

time_varying_known_reals = ["time_idx"]

# 🔥 LIMITED FEATURES (PREVENTS HANG)
time_varying_unknown_reals = [
    "heart_rate",
    "respiratory_rate",
    "spo2_pct",
    "temperature_c",
    "lactate",
    "MAP",
    "pulse_pressure",
    "wbc_count",
    "creatinine",
    "crp_level",
    "NEWS2",
    "sofa_proxy"
]

In [41]:
train_loader = training.to_dataloader(train=True, batch_size=32, num_workers=0)
val_loader = validation.to_dataloader(train=False, batch_size=32, num_workers=0)

In [42]:
tft = TemporalFusionTransformer.from_dataset(
    training,
    learning_rate=1e-3,
    
    hidden_size=16,                # 🔥 reduced
    attention_head_size=1,         # 🔥 reduced
    dropout=0.1,
    hidden_continuous_size=16,     # 🔥 reduced
    
    output_size=2,
    loss=CrossEntropy(),
)

c:\Users\Swanandi\AppData\Local\Programs\Python\Python313\Lib\site-packages\lightning\pytorch\utilities\parsing.py:213: Attribute 'loss' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['loss'])`.
c:\Users\Swanandi\AppData\Local\Programs\Python\Python313\Lib\site-packages\lightning\pytorch\utilities\parsing.py:213: Attribute 'logging_metrics' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['logging_metrics'])`.


In [43]:
early_stop = EarlyStopping(monitor="val_loss", patience=5, mode="min")

trainer = Trainer(
    max_epochs=20,
    accelerator="gpu" if torch.cuda.is_available() else "cpu",
    devices=1,
    
    num_sanity_val_steps=0,   # 🔥 KEY FIX (prevents freeze)
    callbacks=[early_stop],
)

GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.


In [44]:
trainer.fit(tft, train_loader, val_loader)

💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


┏━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃    ┃ Name                               ┃ Type                            ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0  │ loss                               │ CrossEntropy                    │      0 │ train │     0 │
│ 1  │ logging_metrics                    │ ModuleList                      │      0 │ train │     0 │
│ 2  │ input_embeddings                   │ MultiEmbedding                  │     11 │ train │     0 │
│ 3  │ prescalers                         │ ModuleDict                      │  1.8 K │ train │     0 │
│ 4  │ static_variable_selection          │ VariableSelectionNetwork        │  2.6 K │ train │     0 │
│ 5  │ encoder_variable_selection         │ VariableSelectionNetwork        │ 80.1 K │ train │     0 │
│ 6  │ decoder_variable_selection         │ VariableSelectionNetwork        │  1.2 K │ train │     0 │
│ 7  │ static_context_variable_selection  │ GatedResidualNetwork            │  1.1 K │ train │     0 │
│ 8  │ static_context_initial_hidden_lstm │ GatedResidualNetwork            │  1.1 K │ train │     0 │
│ 9  │ static_context_initial_cell_lstm   │ GatedResidualNetwork            │  1.1 K │ train │     0 │
│ 10 │ static_context_enrichment          │ GatedResidualNetwork            │  1.1 K │ train │     0 │
│ 11 │ lstm_encoder                       │ LSTM                            │  2.2 K │ train │     0 │
│ 12 │ lstm_decoder                       │ LSTM                            │  2.2 K │ train │     0 │
│ 13 │ post_lstm_gate_encoder             │ GatedLinearUnit                 │    544 │ train │     0 │
│ 14 │ post_lstm_add_norm_encoder         │ AddNorm                         │     32 │ train │     0 │
│ 15 │ static_enrichment                  │ GatedResidualNetwork            │  1.4 K │ train │     0 │
│ 16 │ multihead_attn                     │ InterpretableMultiHeadAttention │  1.1 K │ train │     0 │
│ 17 │ post_attn_gate_norm                │ GateAddNorm                     │    576 │ train │     0 │
│ 18 │ pos_wise_ff                        │ GatedResidualNetwork            │  1.1 K │ train │     0 │
│ 19 │ pre_output_gate_norm               │ GateAddNorm                     │    576 │ train │     0 │
│ 20 │ output_layer                       │ Linear                          │     34 │ train │     0 │
└────┴────────────────────────────────────┴─────────────────────────────────┴────────┴───────┴───────┘

Trainable params: 98.0 K                                                                                           
Non-trainable params: 0                                                                                            
Total params: 98.0 K                                                                                               
Total estimated model params size (MB): 0                                                                          
Modules in train mode: 787                                                                                         
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

c:\Users\Swanandi\AppData\Local\Programs\Python\Python313\Lib\site-packages\rich\live.py:260: UserWarning: install 
"ipywidgets" for Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')

c:\Users\Swanandi\AppData\Local\Programs\Python\Python313\Lib\site-packages\lightning\pytorch\utilities\_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
c:\Users\Swanandi\AppData\Local\Programs\Python\Python313\Lib\site-packages\lightning\pytorch\trainer\connectors\data_connector.py:434: The 'train_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=11` in the `DataLoader` to improve performance.
c:\Users\Swanandi\AppData\Local\Programs\Python\Python313\Lib\site-packages\lightning\pytorch\trainer\connectors\data_connector.py:434: The 'val_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=11` in the `DataLoader` to improve performance.

Detected KeyboardInterrupt, attempting graceful shutdown ...


SystemExit: 1

c:\Users\Swanandi\AppData\Local\Programs\Python\Python313\Lib\site-packages\IPython\core\interactiveshell.py:3709: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)


In [ ]:
raw_preds = tft.predict(val_loader)

probs = torch.softmax(raw_preds, dim=-1)[:, 1].numpy()
targets = val_df[TARGET].values[:len(probs)]

In [ ]:
from sklearn.metrics import average_precision_score

pr_auc = average_precision_score(targets, probs)

print("Validation PR-AUC:", pr_auc)

In [ ]:
import os

os.makedirs("../models/tft", exist_ok=True)

trainer.save_checkpoint("../models/tft/tft_fold_0.ckpt")